# Loan Application Approval Prediction using K-Nearest Neighbors (KNN)
### Dataset: Kaggle Loan Prediction Dataset (`train.csv`)

This notebook covers the complete machine learning workflow:
1. **Data Preprocessing & Imputation** (Median for numerical, Mode for categorical)
2. **Categorical Feature Encoding**
3. **Train/Test Split (80/20) & Feature Standardization (`StandardScaler`)**
4. **K-Nearest Neighbors Baseline Modeling (`KNeighborsClassifier`)**
5. **Empirical Analysis across multiple $k$ values (3, 5, 7, etc.)**
6. **Hyperparameter Tuning (`n_neighbors`) via 5-Fold Cross Validation**
7. **Model Evaluation & Confusion Matrix Analysis**
8. **Exporting Trained Pipeline for FastAPI REST Service**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

sns.set_theme(style='whitegrid')

## 1. Load Dataset and Initial Inspection

In [ ]:
# Load train.csv
df = pd.read_csv('train.csv')
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
print("Missing values per column before preprocessing:")
print(df.isnull().sum()[df.isnull().sum() > 0])

## 2. Data Preprocessing
- Drop `Loan_ID` (non-predictive unique ID)
- Fill numerical column missing values with **Median**
- Fill categorical column missing values with **Mode**
- Encode categorical variables into numerical values

In [ ]:
data = df.drop(columns=['Loan_ID'])

# Numerical columns: fill with median
num_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']
for col in num_cols:
    data[col] = data[col].fillna(data[col].median())

# Categorical columns: fill with mode
cat_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Credit_History', 'Property_Area']
for col in cat_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

# Verify zero missing values
print("Total missing values remaining:", data.isnull().sum().sum())

In [ ]:
# Encode categorical features into numerical format
encoding_dict = {
    'Gender': {'Female': 0, 'Male': 1},
    'Married': {'No': 0, 'Yes': 1},
    'Dependents': {'0': 0, '1': 1, '2': 2, '3+': 3},
    'Education': {'Graduate': 1, 'Not Graduate': 0},
    'Self_Employed': {'No': 0, 'Yes': 1},
    'Property_Area': {'Rural': 0, 'Semiurban': 1, 'Urban': 2},
    'Loan_Status': {'N': 0, 'Y': 1}
}

for col, mapping in encoding_dict.items():
    data[col] = data[col].map(mapping)

data['Credit_History'] = data['Credit_History'].astype(float)
data['Dependents'] = data['Dependents'].astype(int)
data.head()

## 3. Train-Test Split (80% / 20%) & Feature Scaling
StandardScaler standardizes feature values so distance-based KNN distances are not skewed by high-magnitude variables (like Applicant Income vs Dependents).

In [ ]:
X = data.drop(columns=['Loan_Status'])
y = data['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

## 4. Train Default KNN Classifier & Initial Performance Evaluation

In [ ]:
knn_default = KNeighborsClassifier()  # default n_neighbors=5
knn_default.fit(X_train_scaled, y_train)

y_pred_default = knn_default.predict(X_test_scaled)
acc_default = accuracy_score(y_test, y_pred_default)
cm_default = confusion_matrix(y_test, y_pred_default)

print(f"Default KNN (k=5) Test Accuracy: {acc_default * 100:.2f}%")
print("Confusion Matrix:")
print(cm_default)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_default, target_names=['Rejected', 'Approved']))

## 5. Experiment with Different Values of $k$ (e.g. 3, 5, 7, ...)

In [ ]:
k_values = [1, 3, 5, 7, 9, 11, 15, 17, 21, 25]
results = []

for k in k_values:
    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(X_train_scaled, y_train)
    tr_acc = accuracy_score(y_train, clf.predict(X_train_scaled))
    te_acc = accuracy_score(y_test, clf.predict(X_test_scaled))
    cv_s = cross_val_score(clf, X_train_scaled, y_train, cv=5, scoring='accuracy').mean()
    results.append({'k': k, 'Train Acc': tr_acc, 'Test Acc': te_acc, 'CV 5-Fold': cv_s})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

## 6. Hyperparameter Tuning for `n_neighbors`

In [ ]:
param_grid = {'n_neighbors': list(range(1, 31))}
grid_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)

best_k = grid_search.best_params_['n_neighbors']
print(f"Optimal n_neighbors: {best_k}")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_ * 100:.2f}%")

knn_tuned = grid_search.best_estimator_
y_pred_tuned = knn_tuned.predict(X_test_scaled)
acc_tuned = accuracy_score(y_test, y_pred_tuned)
cm_tuned = confusion_matrix(y_test, y_pred_tuned)

print(f"Tuned KNN (k={best_k}) Test Accuracy: {acc_tuned * 100:.2f}%")
print("Tuned Confusion Matrix:")
print(cm_tuned)
print("\nTuned Classification Report:")
print(classification_report(y_test, y_pred_tuned, target_names=['Rejected', 'Approved']))

## 7. Visualizations: Confusion Matrix & Tuning Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm_default, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Rejected', 'Approved'], yticklabels=['Rejected', 'Approved'])
axes[0].set_title(f'Default KNN (k=5)\nAccuracy: {acc_default*100:.2f}%', fontsize=12)
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

sns.heatmap(cm_tuned, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Rejected', 'Approved'], yticklabels=['Rejected', 'Approved'])
axes[1].set_title(f'Tuned KNN (k={best_k})\nAccuracy: {acc_tuned*100:.2f}%', fontsize=12)
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.suptitle('Confusion Matrix: Default KNN vs Tuned KNN', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()